In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Dashboard UI Setup ---
style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

# Input Widgets
balance_in = widgets.FloatText(value=100000, description='Gross Balance ($):', style=style, layout=layout)
market_pct_in = widgets.FloatSlider(value=15, min=0, max=100, step=0.5, description='Market Price (%):', style=style)
costs_in = widgets.FloatText(value=2000, description='Transaction Costs ($):', style=style, layout=layout)
discount_in = widgets.FloatSlider(value=10, min=1, max=30, step=0.5, description='Discount Rate (%):', style=style)
prob_in = widgets.FloatSlider(value=65, min=0, max=100, step=1, description='Collection Prob (%):', style=style)
years_in = widgets.IntSlider(value=5, min=1, max=20, description='Term (Years):', style=style)

# Run Button
run_button = widgets.Button(description="Rerun Model", button_style='success', icon='refresh')
output = widgets.Output()

# --- 2. Financial Logic ---
def run_liquidation_model(b):
    with output:
        clear_output(wait=True)
        
        # Capture Inputs
        balance = balance_in.value
        mkt_price = market_pct_in.value / 100
        costs = costs_in.value
        rate = discount_in.value / 100
        prob = prob_in.value / 100
        term = years_in.value
        
        # NPV Liquidation Calculation (Immediate Cash)
        # Represents the absolute monetary value of selling today
        npv_liq = (balance * mkt_price) - costs
        
        # DCF Life Time Liquidation Calculation
        # Future cash flows are worth less than dollars today due to time value
        annual_nominal_cf = (balance / term) * prob
        dcf_data = []
        lifetime_pv = 0
        
        for year in range(1, term + 1):
            pv = annual_nominal_cf / (1 + rate)**year
            lifetime_pv += pv
            dcf_data.append({
                "Year": year, 
                "Expected Cash Flow": f"${annual_nominal_cf:,.2f}", 
                "Present Value": f"${pv:,.2f}"
            })
        
        # Strategy Recommendation
        recommendation = "Life Time Liquidation" if lifetime_pv > npv_liq else "NPV Liquidation (Sale)"
        delta = abs(lifetime_pv - npv_liq)
        
        # Visualization
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(['Immediate NPV Sale', 'Life Time DCF'], [npv_liq, lifetime_pv], color=['#e74c3c', '#3498db'])
        ax.set_title(f'Strategy Recommendation: {recommendation}', fontsize=14, fontweight='bold')
        ax.set_ylabel('Current Value ($)')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Display Results
        print(f"{'='*40}")
        print(f"DECISION: {recommendation}")
        print(f"{'='*40}")
        print(f"Net Present Value (Sale):  ${npv_liq:,.2f}")
        print(f"Life Time DCF (Collect):  ${lifetime_pv:,.2f}")
        print(f"Incremental Gain:         ${delta:,.2f}")
        print(f"\n--- DCF Schedule ---")
        display(pd.DataFrame(dcf_data).set_index("Year"))
        plt.show()

# --- 3. Interaction Setup ---
run_button.on_click(run_liquidation_model)

# Layout the Dashboard
input_box = widgets.VBox([
    widgets.HTML("<b>Portfolio Assumptions</b>"),
    balance_in, market_pct_in, costs_in, 
    widgets.HTML("<br><b>Collection Assumptions</b>"),
    discount_in, prob_in, years_in,
    run_button
])

display(widgets.HBox([input_box, output]))

# Initial Run
run_liquidation_model(None)